# Tests based on UGC 7346

In [ ]:
#%matplotlib ipympl
from matplotlib import pyplot as plt
from matplotlib import colors
from matplotlib.ticker import AutoMinorLocator

import numpy as np
from scipy import ndimage

from astropy.table import Table, vstack
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy import constants as c

from pst.observables import load_photometric_filters

# 1. Init

## Plotting

In [ ]:
def new_figure(fig_name, figsize=None, nrows=1, ncols=1, sharex='col', sharey='row', gridspec_kw={'hspace': 0, 'wspace': 0}, **subplot_kwargs):
    if figsize is None:
        figsize = (9 + ncols, 4 + nrows)
        
    plt.close(fig_name)
    fig = plt.figure(fig_name, figsize=figsize)
    axes = fig.subplots(nrows=nrows, ncols=ncols, squeeze=False,
                        sharex=sharex, sharey=sharey,
                        gridspec_kw=gridspec_kw,
                        **subplot_kwargs
                       )
    fig.set_tight_layout(True)
    for ax in axes.flat:
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which='both', bottom=True, top=True, left=True, right=True)
        ax.tick_params(which='major', direction='inout', length=8, grid_alpha=.3)
        ax.tick_params(which='minor', direction='in', length=2, grid_alpha=.1)
        ax.grid(True, which='both')

    #fig.suptitle(f'{os.path.basename(rss.filename)}: {fig_name}')  # VERY dirty hack :^(
    fig.suptitle(fig_name)
    
    return fig, axes


## Reading WEAVE data

In [ ]:
class WEAVE_RSS(object):
    
    def __init__(self, filename):
        '''Read a WEAVE "single exposure" file (i.e. row-stacked spectra for just one arm)'''
        self.filename = filename
        self.hdu = fits.open(filename)
        self.wcs = WCS(self.hdu[1].header)
        pixels = np.arange(self.hdu[1].data.shape[1])
        self.wavelength = self.wcs.spectral.array_index_to_world(pixels).to_value(u.Angstrom)
        self.sensitivity_function = self.hdu[5].data
        self.intensity = self.hdu[1].data * self.sensitivity_function * 1e17/(np.pi*1.305**2)  # [1e-17 erg/s/cm^2/Angstrom/arcsec^2]
        self.error = 1 / np.sqrt(np.ma.masked_array(self.hdu[2].data, self.hdu[2].data > 0))
        self.fibtable = self.hdu[6].data
        self.position = SkyCoord(ra=self.fibtable["FIBRERA"]<<u.deg, dec=self.fibtable["FIBREDEC"]<<u.deg)
        self.median_intensity = np.nanmedian(self.intensity, axis=1)
        self.sky_spectrum = self.hdu[3].data[0] * self.sensitivity_function[0] * 1e17/(np.pi*1.305**2) - self.intensity[0]
        
red_arm = [WEAVE_RSS("data/UGC7346/2025B2/21516/L1/single_3134311.fit"),
           WEAVE_RSS("data/UGC7346/2025B2/21516/L1/single_3134313.fit"),
           WEAVE_RSS("data/UGC7346/2025B2/21516/L1/single_3134315.fit"),
           WEAVE_RSS("data/UGC7346/2025B2/21517/L1/single_3134335.fit"),
           WEAVE_RSS("data/UGC7346/2025B2/21517/L1/single_3134337.fit"),
           WEAVE_RSS("data/UGC7346/2025B2/21517/L1/single_3134339.fit"),
           WEAVE_RSS("data/UGC7346/2025B2/21518/L1/single_3133922.fit"),
           WEAVE_RSS("data/UGC7346/2025B2/21518/L1/single_3133924.fit"),
           WEAVE_RSS("data/UGC7346/2025B2/21518/L1/single_3133926.fit"),
          ]
blue_arm  = [WEAVE_RSS("data/UGC7346/2025B2/21516/L1/single_3134312.fit"),
             WEAVE_RSS("data/UGC7346/2025B2/21516/L1/single_3134314.fit"),
             WEAVE_RSS("data/UGC7346/2025B2/21516/L1/single_3134316.fit"),
             WEAVE_RSS("data/UGC7346/2025B2/21517/L1/single_3134336.fit"),
             WEAVE_RSS("data/UGC7346/2025B2/21517/L1/single_3134338.fit"),
             WEAVE_RSS("data/UGC7346/2025B2/21517/L1/single_3134340.fit"),
             WEAVE_RSS("data/UGC7346/2025B2/21518/L1/single_3133923.fit"),
             WEAVE_RSS("data/UGC7346/2025B2/21518/L1/single_3133925.fit"),
             WEAVE_RSS("data/UGC7346/2025B2/21518/L1/single_3133927.fit"),
            ]

## Target params

In [ ]:
target_centre = SkyCoord(184.6742477*u.deg, 17.7183503*u.deg)
target_redshift = 0.00231
fibre_distance = []
for arm in red_arm:
    fibre_distance.append(target_centre.separation(arm.position))

# 2. Plots

## Regions

In [ ]:
n_regions = 4
radial_edges = np.linspace(0, 40, n_regions+1) << u.arcsec
region_mask = []
for i in range(n_regions):
    region_mask_pointing = []
    for arm in red_arm:
        region_mask_pointing.append((target_centre.separation(arm.position) > radial_edges[i]) & (target_centre.separation(arm.position) < radial_edges[i+1]))
    region_mask.append(region_mask_pointing)
region_mask = np.array(region_mask)

In [ ]:
fig, axes = new_figure("regions", figsize=(12, 3*n_regions), nrows=n_regions)

def plot_region(arm_list, mask_list, color):
    stack = []
    for arm, mask in zip(arm_list, mask_list):
        mean = np.nanmean(arm.intensity[mask], axis=0)
        stack.append(mean)
        ax.plot(arm.wavelength, mean, f'{color}-', alpha=.15)
        '''
        p16, p50, p84 = np.nanpercentile(arm.intensity[mask], [16, 50, 84], axis=0)
        ax.plot(arm.wavelength, p50, 'k-', alpha=.5)
        ax.fill_between(arm.wavelength, p16, p84, color='k', alpha=.1)
        #n_fibres = np.count_nonzero(mask)
        #ax.fill_between(wavelength, p50-(p50-p16)/np.sqrt(n_fibres), p50+(p84-p50)/np.sqrt(n_fibres), color='k', alpha=.1)
        '''

        '''
        smooth = ndimage.median_filter(mean, 100)
        intenisty_ab = 3631 / (3.34e4 * arm.wavelength**2)
        mu_ab = np.nanmedian(-2.5*np.log10(1e-17*smooth[smooth > 0] / intenisty_ab[smooth > 0]))
        ax.plot(arm.wavelength, 1e17*intenisty_ab*10**(-.4*mu_ab), f'{color}:', alpha=.5, label=f'$\\mu_{{AB}} = {mu_ab:.2f}$ mag/arcsec$^2$')
        '''

        #ax.plot(arm.wavelength, arm.sky_spectrum, 'g-', alpha=.25)
    
    stack_mean = np.nanmean(stack, axis=0)
    stack_std = np.nanstd(stack, axis=0)
    ax.plot(arm.wavelength, stack_mean, 'k-', alpha=1)
    ax.plot(arm.wavelength, stack_mean+stack_std, 'k:', alpha=1)
    ax.plot(arm.wavelength, stack_mean-stack_std, 'k:', alpha=1)
    return Table([arm.wavelength, stack_mean, stack_std], names=("wavelength", "spectrum", "error"))


red_tables = []
blue_tables = []
for i in range(n_regions):
    ax = axes[i, 0]
    ax.set_ylabel(r"$I_\lambda$ [$10^{-17}$ erg/s/cm$^2/\AA$/arcsec$^2$]")
    red_tables.append(plot_region(red_arm, region_mask[i], 'r'))
    blue_tables.append(plot_region(blue_arm, region_mask[i], 'b'))
    
    #p5_blue, p50_blue, p95_blue = positive_percentiles(blue_arm.intensity[region_mask[i]])
    #p5_red, p50_red, p95_red = positive_percentiles(red_arm.intensity[region_mask[i]])
    #ax.set_ylim(-min(p5_blue, p5_red), min(np.max([p95_blue, p95_red, p50_blue+p50_red]), 2*(p50_blue+p50_red)))
    #ax.set_ylim(0, 1.5)
    ax.set_ylim(-.05, .65)
    ax.legend(title=f'{radial_edges[i].to(u.arcsec):.2f} < r < {radial_edges[i+1].to(u.arcsec):.2f}')
    red_tables[-1].write(f'output/red_{radial_edges[i].to_value(u.arcsec):02.0f}_{radial_edges[i+1].to_value(u.arcsec):02.0f}.csv', overwrite=True)
    blue_tables[-1].write(f'output/blue_{radial_edges[i].to_value(u.arcsec):02.0f}_{radial_edges[i+1].to_value(u.arcsec):02.0f}.csv', overwrite=True)

ax = axes[-1, 0]
ax.set_xlabel(r"$\lambda$ [$\AA$]")
#ax.set_xlim(3600, 4200)
#ax.set_xlim(4000, 4500)
#ax.set_xlim(4500, 5000)
#ax.set_xlim(6000, 6700)
'''
for ax in axes[:, 0]:
    ax.set_xlim(3800, 4500)
    ax.axvline(3934.777*(1+target_redshift), c='k', ls=':')
    ax.axvline(3969.588*(1+target_redshift), c='k', ls=':')
    ax.axvline(4101.734*(1+target_redshift), c='k', ls=':')
    ax.axvline(4340.472*(1+target_redshift), c='k', ls=':')
    ax.axvline(4305.61*(1+target_redshift), c='k', ls=':')
    ax.axvline(4364.436*(1+target_redshift), c='k', ls=':')
'''

'''
for ax in axes[:, 0]:
    ax.set_xlim(4800, 5300)
    ax.axvline(4861.35*(1+target_redshift), c='k', ls=':')
    ax.axvline(4960.295*(1+target_redshift), c='k', ls=':')
    ax.axvline(5008.240*(1+target_redshift), c='k', ls=':')
    ax.axvline(5184 *(1+target_redshift), c='k', ls=':')
    ax.axvline(5173 *(1+target_redshift), c='k', ls=':')
    ax.axvline(5167 *(1+target_redshift), c='k', ls=':')
'''

for ax in axes[:, 0]:
    ax.set_xlim(6500, 6800)
    ax.axvline(6549.86 *(1+target_redshift), c='k', ls=':')
    ax.axvline(6562.79*(1+target_redshift), c='k', ls=':')
    ax.axvline(6585.27 *(1+target_redshift), c='k', ls=':')
    ax.axvline(6718.29 *(1+target_redshift), c='k', ls=':')
    ax.axvline(6732.67 *(1+target_redshift), c='k', ls=':')
'''
'''

'''
for ax in axes[:, 0]:
    ax.set_xlim(8000, 9000)
    ax.axvline(8498*(1+target_redshift), c='k', ls=':')
    ax.axvline(8542*(1+target_redshift), c='k', ls=':')
    ax.axvline(8662*(1+target_redshift), c='k', ls=':')
'''
